In [1]:
# Load environment variables from project root
from pathlib import Path
from dotenv import load_dotenv

for _dir in [Path.cwd(), *Path.cwd().parents]:
    _env = _dir / '.env'
    if _env.is_file():
        load_dotenv(_env)
        break

In [2]:
# Create Databricks Spark session
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()
spark.sql('CREATE SCHEMA IF NOT EXISTS ddc_databricks.silver')

DataFrame[]

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType, ArrayType

# Read bronze source tables
bronze_questions = spark.table('ddc_databricks.bronze.so_questions')
bronze_answers = spark.table('ddc_databricks.bronze.so_answers')
bronze_tag_info = spark.table('ddc_databricks.bronze.so_tag_info')

In [5]:
# 1) Questions -> silver.so_questions
html_code_pattern = F.lit(r'(?is)<pre><code>.*?</code></pre>')
html_link_pattern = F.lit(r'(?is)<a\s+[^>]*href=')

questions_flat = (
    bronze_questions
    .select(
        F.col('_pypi_package').alias('package_name'),
        F.col('_so_tag').alias('so_tag'),
        F.to_timestamp('_ingested_at').alias('ingested_at'),
        F.col('_endpoint').alias('source_endpoint'),
        F.col('_source').alias('source_system'),
        F.explode_outer('questions').alias('q')
    )
    .withColumn('q_json', F.to_json(F.col('q')))
    .select(
        'package_name',
        'so_tag',
        'ingested_at',
        'source_endpoint',
        'source_system',
        F.col('q.question_id').cast('long').alias('question_id'),
        F.from_unixtime(F.col('q.creation_date')).cast('timestamp').alias('question_created_at'),
        F.from_unixtime(F.col('q.last_activity_date')).cast('timestamp').alias('question_last_activity_at'),
        F.from_unixtime(F.col('q.last_edit_date')).cast('timestamp').alias('question_last_edit_at'),
        F.from_unixtime(F.col('q.closed_date')).cast('timestamp').alias('question_closed_at'),
        F.col('q.score').cast('long').alias('question_score'),
        F.col('q.view_count').cast('long').alias('question_view_count'),
        F.col('q.answer_count').cast('long').alias('question_answer_count'),
        F.get_json_object(F.col('q_json'), '$.favorite_count').cast('long').alias('question_favorite_count'),
        F.col('q.is_answered').cast('boolean').alias('is_answered'),
        F.col('q.tags').alias('question_tags'),
        F.col('q.title').alias('question_title'),
        F.col('q.body').alias('question_body_html'),
        F.col('q.owner.user_id').cast('long').alias('owner_user_id'),
        F.col('q.owner.display_name').alias('owner_display_name'),
        F.col('q.owner.reputation').cast('long').alias('owner_reputation')
    )
    .filter(F.col('question_id').isNotNull())
    .withColumn('question_title_char_count', F.length(F.coalesce(F.col('question_title'), F.lit(''))))
    .withColumn('question_body_char_count', F.length(F.coalesce(F.col('question_body_html'), F.lit(''))))
    .withColumn('question_body_code_block_count', F.regexp_count(F.col('question_body_html'), html_code_pattern))
    .withColumn('question_body_link_count', F.regexp_count(F.col('question_body_html'), html_link_pattern))
    .withColumn('question_tags_count', F.size(F.coalesce(F.col('question_tags'), F.array())))
)

questions_dedup_w = Window.partitionBy('package_name', 'question_id').orderBy(F.col('ingested_at').desc(), F.col('question_last_activity_at').desc())
questions_latest = (
    questions_flat
    .withColumn('rn', F.row_number().over(questions_dedup_w))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

(
    questions_latest
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.so_questions')
)

print('silver.so_questions created')

silver.so_questions created


In [10]:
# 2) Answers -> silver.so_answers
html_code_pattern = F.lit(r'(?is)<pre><code>.*?</code></pre>')
html_link_pattern = F.lit(r'(?is)<a\s+[^>]*href=')

answers_flat = (
    bronze_answers
    .select(
        F.col('_pypi_package').alias('package_name'),
        F.col('_so_tag').alias('so_tag'),
        F.to_timestamp('_ingested_at').alias('ingested_at'),
        F.col('_endpoint').alias('source_endpoint'),
        F.col('_source').alias('source_system'),
        F.explode_outer('answers').alias('a')
    )
    .select(
        'package_name',
        'so_tag',
        'ingested_at',
        'source_endpoint',
        'source_system',
        F.col('a.answer_id').cast('long').alias('answer_id'),
        F.col('a.question_id').cast('long').alias('question_id'),
        F.from_unixtime(F.col('a.creation_date')).cast('timestamp').alias('answer_created_at'),
        F.from_unixtime(F.col('a.last_activity_date')).cast('timestamp').alias('answer_last_activity_at'),
        F.from_unixtime(F.col('a.last_edit_date')).cast('timestamp').alias('answer_last_edit_at'),
        F.col('a.score').cast('long').alias('answer_score'),
        F.col('a.is_accepted').cast('boolean').alias('is_accepted'),
        F.col('a.body').alias('answer_body_html'),
        F.col('a.owner.user_id').cast('long').alias('owner_user_id'),
        F.col('a.owner.display_name').alias('owner_display_name'),
        F.col('a.owner.reputation').cast('long').alias('owner_reputation')
    )
    .filter(F.col('answer_id').isNotNull())
    .withColumn('answer_body_char_count', F.length(F.coalesce(F.col('answer_body_html'), F.lit(''))))
    .withColumn('answer_body_code_block_count', F.regexp_count(F.col('answer_body_html'), html_code_pattern))
    .withColumn('answer_body_link_count', F.regexp_count(F.col('answer_body_html'), html_link_pattern))
)

answers_dedup_w = Window.partitionBy('package_name', 'answer_id').orderBy(F.col('ingested_at').desc(), F.col('answer_last_activity_at').desc())
answers_latest = (
    answers_flat
    .withColumn('rn', F.row_number().over(answers_dedup_w))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

(
    answers_latest
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.so_answers')
)

print('silver.so_answers created')

silver.so_answers created


In [11]:
# 3) Tag info -> silver.so_tag_info
tag_item_schema = ArrayType(StructType([
    StructField('name', StringType(), True),
    StructField('count', LongType(), True),
    StructField('has_synonyms', BooleanType(), True),
    StructField('is_moderator_only', BooleanType(), True),
    StructField('is_required', BooleanType(), True)
]))

tag_info_clean = (
    bronze_tag_info
    .select(
        F.col('_pypi_package').alias('package_name'),
        F.col('_so_tag').alias('so_tag'),
        F.to_timestamp('_ingested_at').alias('ingested_at'),
        F.col('_endpoint').alias('source_endpoint'),
        F.col('_source').alias('source_system'),
        F.to_json(F.col('data')).alias('data_json')
    )
    .withColumn('items_json', F.coalesce(F.get_json_object(F.col('data_json'), '$.items'), F.lit('[]')))
    .withColumn('items', F.from_json(F.col('items_json'), tag_item_schema))
    .withColumn('item', F.explode_outer('items'))
    .select(
        'package_name',
        'so_tag',
        'ingested_at',
        'source_endpoint',
        'source_system',
        F.coalesce(F.col('item.name'), F.col('so_tag')).alias('tag_name'),
        F.col('item.count').cast('long').alias('tag_question_count'),
        F.col('item.has_synonyms').cast('boolean').alias('has_synonyms'),
        F.col('item.is_moderator_only').cast('boolean').alias('is_moderator_only'),
        F.col('item.is_required').cast('boolean').alias('is_required')
    )
)

tag_info_dedup_w = Window.partitionBy('package_name', 'tag_name').orderBy(F.col('ingested_at').desc())
tag_info_latest = (
    tag_info_clean
    .withColumn('rn', F.row_number().over(tag_info_dedup_w))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

(
    tag_info_latest
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.so_tag_info')
)

print('silver.so_tag_info created')

silver.so_tag_info created


In [12]:
# 4) Cross-linked package-level Stack Overflow features -> silver.so_package_features
question_features = (
    questions_latest
    .groupBy('package_name')
    .agg(
        F.countDistinct('question_id').alias('questions_total'),
        F.sum(F.when(F.col('is_answered') == F.lit(True), 1).otherwise(0)).alias('answered_questions_total'),
        F.avg('question_score').alias('avg_question_score'),
        F.avg('question_view_count').alias('avg_question_views'),
        F.avg('question_answer_count').alias('avg_answers_per_question'),
        F.sum(F.coalesce(F.col('question_favorite_count'), F.lit(0))).alias('question_favorites_total'),
        F.countDistinct('owner_user_id').alias('unique_question_askers'),
        F.max('question_created_at').alias('latest_question_at'),
        F.sum(F.when(F.col('question_created_at') >= F.expr('current_timestamp() - INTERVAL 30 DAYS'), 1).otherwise(0)).alias('questions_30d')
    )
    .withColumn('question_answered_rate', F.when(F.col('questions_total') > 0, F.col('answered_questions_total') / F.col('questions_total')).otherwise(F.lit(None)))
)

answer_features = (
    answers_latest
    .groupBy('package_name')
    .agg(
        F.countDistinct('answer_id').alias('answers_total'),
        F.sum(F.when(F.col('is_accepted') == F.lit(True), 1).otherwise(0)).alias('accepted_answers_total'),
        F.avg('answer_score').alias('avg_answer_score'),
        F.countDistinct('owner_user_id').alias('unique_answerers'),
        F.max('answer_created_at').alias('latest_answer_at'),
        F.sum(F.when(F.col('answer_created_at') >= F.expr('current_timestamp() - INTERVAL 30 DAYS'), 1).otherwise(0)).alias('answers_30d')
    )
    .withColumn('accepted_answer_rate', F.when(F.col('answers_total') > 0, F.col('accepted_answers_total') / F.col('answers_total')).otherwise(F.lit(None)))
)

tag_features = (
    tag_info_latest
    .groupBy('package_name')
    .agg(
        F.max('tag_question_count').alias('tag_total_questions'),
        F.max(F.when(F.col('has_synonyms') == F.lit(True), F.lit(1)).otherwise(F.lit(0))).alias('tag_has_synonyms_flag'),
        F.max(F.when(F.col('is_required') == F.lit(True), F.lit(1)).otherwise(F.lit(0))).alias('tag_is_required_flag')
    )
)

so_package_features = (
    question_features.alias('q')
    .join(answer_features.alias('a'), on='package_name', how='full')
    .join(tag_features.alias('t'), on='package_name', how='left')
    .withColumn('qa_ratio', F.when(F.col('questions_total') > 0, F.col('answers_total') / F.col('questions_total')).otherwise(F.lit(None)))
    .withColumn('community_activity_30d', F.coalesce(F.col('questions_30d'), F.lit(0)) + F.coalesce(F.col('answers_30d'), F.lit(0)))
)

(
    so_package_features
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.so_package_features')
)

print('silver.so_package_features created')

silver.so_package_features created


In [ ]:
# 5) Basic data quality checks for Silver Stack Overflow outputs
quality_metrics = [
    ('bronze_so_questions_rows', bronze_questions.count()),
    ('bronze_so_answers_rows', bronze_answers.count()),
    ('bronze_so_tag_info_rows', bronze_tag_info.count()),
    ('silver_so_questions_rows', questions_latest.count()),
    ('silver_so_answers_rows', answers_latest.count()),
    ('silver_so_tag_info_rows', tag_info_latest.count()),
    ('silver_so_package_features_rows', so_package_features.count()),
    ('silver_so_questions_missing_body', questions_latest.filter(F.col('question_body_html').isNull()).count()),
    ('silver_so_answers_missing_body', answers_latest.filter(F.col('answer_body_html').isNull()).count()),
    ('silver_so_unmatched_answers_to_questions', answers_latest.alias('a').join(questions_latest.alias('q'), on=[F.col('a.package_name') == F.col('q.package_name'), F.col('a.question_id') == F.col('q.question_id')], how='left').filter(F.col('q.question_id').isNull()).count())
]

quality_df = spark.createDataFrame(quality_metrics, ['metric_name', 'metric_value']).withColumn(
    'checked_at', F.current_timestamp()
)

(
    quality_df
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.so_quality_checks')
)

quality_df.orderBy('metric_name').show(truncate=False)